# Biomechanische Validatie: AI vs. Sensor Data

Kwantitatieve vergelijking tussen de door het model berekende SPM en de grondwaarheid van de accelerometers.

> **Vereisten:**
> - Detectiebestand `P_J16.json`
> - Sensor data `P_J16.h5`

### Initialisatie en Data-Ingestie

In [ ]:
import os
import json
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error
from itertools import groupby
from operator import itemgetter

# --- PADEN ---
FILE_J16 = '../../data/ground_truth/dataset_C/P_J16.h5'
DATA_JSON_PATH = '../../data/detecties/P_J16.json'
OUTPUT_DIR = '../../data/resultaten/' # Zorg dat deze map bestaat of pas aan

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

if os.path.exists(DATA_JSON_PATH):
    with open(DATA_JSON_PATH, 'r') as f:
        loaded_data = json.load(f)
        runner_data = {int(k): v for k, v in loaded_data['tracks'].items()}
        fps_ai = loaded_data['fps']
    print(f"Ruwe AI-tracking data geladen! ({len(runner_data)} lopers)")
else:
    print(f"Bestand niet gevonden: {DATA_JSON_PATH}.")

### Kwantitatieve Validatie en Bland-Altman Analyse

In [ ]:
# --- INSTELLINGEN ---
TRIM_SEC = 5.0
FS_AI = float(fps_ai)
FS_SENSOR = 200.0

mapping = {
    'P_1': [1], 'P_17': [7], 'P_36': [2], 'P_20': [11], 'P_41': [6],
    'P_58': [15], 'P_59': [30], 'P_61': [9], 'P_6': [10], 'P_8': [13],
    'P_31': [3], 'P_48': [5], 'P_50': [4], 'P_67': [12],
    'P_12': [8, 17, 18, 20, 26, 28],
    'P_66': [14, 19, 21, 22, 24, 25, 27]
}

def get_sensor_spm(ped_id_str):
    try:
        with h5py.File(FILE_J16, 'r') as f:
            p_ids_bytes = [pid.decode('utf-8') for pid in np.array(f['ped_ID']).squeeze()]
            if ped_id_str not in p_ids_bytes: return None

            idx = p_ids_bytes.index(ped_id_str)
            acc = np.array(f['acc_LB_vert'])[:, idx]
            time_arr = np.array(f['time']).squeeze()
            rel_time = time_arr - time_arr[0]

            totale_tijd = rel_time[-1]
            if totale_tijd > (TRIM_SEC * 2):
                mask = (rel_time >= TRIM_SEC) & (rel_time <= (totale_tijd - TRIM_SEC))
                acc = acc[mask]
                time_arr = time_arr[mask]

            b, a = butter(4, 3.0 / (0.5 * FS_SENSOR), btype='low')
            acc_f = filtfilt(b, a, acc)
            peaks, _ = find_peaks(acc_f, height=np.mean(acc_f), distance=FS_SENSOR/4)

            if len(peaks) > 1:
                return ( (len(peaks)-1) / (time_arr[peaks[-1]] - time_arr[peaks[0]]) ) * 60
    except Exception as e:
        pass
    return None

def get_ai_spm(t_id):
    if t_id not in runner_data: return None, 0

    y_coords = np.array(runner_data[t_id]['y_coords'])
    # Herbereken tijd op basis van frame index en FPS
    frames = np.array(runner_data[t_id]['frames'])
    time_stamps = frames / FS_AI
    rel_time = time_stamps - time_stamps[0]

    totale_tijd = rel_time[-1]

    if totale_tijd > (TRIM_SEC * 2) + 2.0:
        mask = (rel_time >= TRIM_SEC) & (rel_time <= (totale_tijd - TRIM_SEC))
        y_coords = y_coords[mask]
        time_stamps = time_stamps[mask]

    if len(y_coords) <= 15: return None, 0

    try:
        b, a = butter(2, [0.8 / (0.5 * FS_AI), 2.0 / (0.5 * FS_AI)], btype='band')
        y_f = filtfilt(b, a, y_coords)
        peaks, _ = find_peaks(y_f, distance=FS_AI*0.4)

        if len(peaks) > 1:
            p2p_time = time_stamps[peaks[-1]] - time_stamps[peaks[0]]
            if p2p_time > 0:
                spm = ((len(peaks) - 1) * 2 / p2p_time) * 60
                return spm, (time_stamps[-1] - time_stamps[0])
    except:
        pass
    return None, 0

comparison_data = []
print(f"{'AI ID':<10} | {'Sensor SPM':<12} | {'AI SPM (Avg)':<12} | {'Afwijking'}")
print("-" * 55)

for p_id, tracker_list in mapping.items():
    sensor_spm = get_sensor_spm(p_id)
    if sensor_spm is None or pd.isna(sensor_spm):
        continue

    ai_spms = []
    track_durations = []

    for t_id in tracker_list:
        spm, dur = get_ai_spm(t_id)
        if spm is not None and 80 <= spm <= 240:
            ai_spms.append(spm)
            track_durations.append(dur)

    if len(ai_spms) > 0:
        weighted_ai_spm = np.average(ai_spms, weights=track_durations)
        diff = abs(sensor_spm - weighted_ai_spm)
        comparison_data.append({'ID': p_id, 'Sensor': sensor_spm, 'AI': weighted_ai_spm, 'Error': diff})
        print(f"{p_id:<10} | {sensor_spm:<12.1f} | {weighted_ai_spm:<12.1f} | {diff:.1f}")

df_val = pd.DataFrame(comparison_data)

if not df_val.empty:
    mae = mean_absolute_error(df_val['Sensor'], df_val['AI'])
    rmse = np.sqrt(mean_squared_error(df_val['Sensor'], df_val['AI']))

    print("\n" + "="*50)
    print(f"FINALE VALIDATIE METRIEK (Steady-State Fase)")
    print(f"Gemiddelde Absolute Fout (MAE): {mae:.2f} SPM")
    print(f"Root Mean Square Error (RMSE): {rmse:.2f} SPM")
    print("="*50)

    df_val['Mean'] = (df_val['Sensor'] + df_val['AI']) / 2
    df_val['Diff'] = df_val['AI'] - df_val['Sensor']
    md = df_val['Diff'].mean()
    sd = df_val['Diff'].std()

    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_val, x='Mean', y='Diff', s=100, color='blue', edgecolor='black')
    plt.axhline(md, color='red', linestyle='--', label=f'Bias: {md:.2f} SPM')
    plt.axhline(md + 1.96*sd, color='gray', linestyle=':', label='95% Upper')
    plt.axhline(md - 1.96*sd, color='gray', linestyle=':', label='95% Lower')

    plt.title('Bland-Altman Plot: AI vs. Sensor', fontsize=14, fontweight='bold')
    plt.xlabel('Gemiddelde van AI en Sensor (SPM)')
    plt.ylabel('Verschil (AI - Sensor) (SPM)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

### Casestudy: Signaalvergelijking tussen Sensor en AI-Tracker

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans', 'Arial', 'Helvetica', 'sans-serif']

FONT_SIZE_LABELS = 16
FONT_SIZE_TICKS = 14
FONT_SIZE_LEGEND = 14
FONT_SIZE_TITLE = 18

TARGET_AI_ID = 11
TARGET_SENSOR_ID = 'P_20'
ZOOM_START_SEC = 20.0
ZOOM_DURATION = 20.0

print(f"Genereren casestudy figuur: Tracker #{TARGET_AI_ID} vs Sensor {TARGET_SENSOR_ID}")

# Sensor
with h5py.File(FILE_J16, 'r') as f:
    p_ids = [pid.decode('utf-8') for pid in np.array(f['ped_ID']).squeeze()]
    idx = p_ids.index(TARGET_SENSOR_ID)
    acc = np.array(f['acc_LB_vert'])[:, idx]
    time_arr = np.array(f['time']).squeeze()

    rel_time = time_arr - time_arr[0]
    mask_s = (rel_time >= ZOOM_START_SEC) & (rel_time <= ZOOM_START_SEC + ZOOM_DURATION)
    time_s = rel_time[mask_s] - ZOOM_START_SEC
    acc_zoom = acc[mask_s]

b_s, a_s = butter(4, 3.0 / (0.5 * FS_SENSOR), btype='low')
acc_f = filtfilt(b_s, a_s, acc_zoom)
peaks_s, _ = find_peaks(acc_f, height=np.mean(acc_f)*1.2, distance=FS_SENSOR/4)

spm_sensor = ((len(peaks_s) - 1) / (time_s[-1] - time_s[0])) * 60 if len(peaks_s) > 1 else np.nan

# AI
y_coords = np.array(runner_data[TARGET_AI_ID]['y_coords'])
frames = np.array(runner_data[TARGET_AI_ID]['frames'])
time_stamps_ai = frames / FS_AI
rel_time_ai = time_stamps_ai - time_stamps_ai[0]

mask_ai = (rel_time_ai >= ZOOM_START_SEC) & (rel_time_ai <= ZOOM_START_SEC + ZOOM_DURATION)
time_ai = rel_time_ai[mask_ai] - ZOOM_START_SEC
y_zoom = y_coords[mask_ai]

b_ai, a_ai = butter(2, [0.8/(0.5*FS_AI), 2.0/(0.5*FS_AI)], btype='band')
y_f = filtfilt(b_ai, a_ai, y_zoom)
peaks_ai, _ = find_peaks(y_f, distance=FS_AI*0.4)

spm_ai = ((len(peaks_ai) - 1) / (time_ai[-1] - time_ai[0])) * 60 * 2 if len(peaks_ai) > 1 else np.nan

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'hspace': 0.15})
fig.suptitle(f'Casestudy: sensor P_{TARGET_SENSOR_ID[2:]} vs. tracker #{TARGET_AI_ID}',
             fontsize=FONT_SIZE_TITLE + 1, fontweight='bold', y=0.96)

ax1.plot(time_s, acc_f, color='#2c3e50', linewidth=1.5, label='Gefilterde verticale acceleratie')
ax1.scatter(time_s[peaks_s], acc_f[peaks_s], color='#d62728', s=30, edgecolor='white', linewidth=0.8, label='Gedetecteerde voetlandingen', zorder=5)
ax1.set_ylim(-25,30)
ax1.set_title(f'Sensor - cadans: {spm_sensor:.1f}'.replace('.', ',') + ' SPM', fontsize=FONT_SIZE_TITLE)
ax1.set_ylabel('Versnelling (m/s²)', fontsize=FONT_SIZE_LABELS)

ax2.plot(time_ai, y_f, color='#1f77b4', linewidth=1.5, label='Gefilterde laterale sway')
ax2.scatter(time_ai[peaks_ai], y_f[peaks_ai], color='#D55E00', s=30, edgecolor='white', linewidth=0.8, label='Gedetecteerde schrede', zorder=5)
ax2.set_ylim(-15,15)
ax2.set_title(f'Model - cadans: {spm_ai:.1f}'.replace('.', ',') + ' SPM', fontsize=FONT_SIZE_TITLE)
ax2.set_ylabel('Y-oscillatie (pixels)', fontsize=FONT_SIZE_LABELS)
ax2.set_xlabel('Tijd (s)', fontsize=FONT_SIZE_LABELS)

for ax in [ax1, ax2]:
    ax.set_xlim(0, 20.02)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, direction='out', top=False, right=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#1a1a1a')
    ax.spines['bottom'].set_color('#1a1a1a')
    ax.grid(True, which='both', linestyle='-', color='#B0B0B0', linewidth=0.4, alpha=0.5)
    leg = ax.legend(fontsize=FONT_SIZE_LEGEND, loc='upper right', frameon=True, framealpha=1, facecolor='white', edgecolor='black')
    leg.get_frame().set_linewidth(0.5)

plt.tight_layout()
plot_pad = os.path.join(OUTPUT_DIR, f'Signaal_Vergelijking_AI{TARGET_AI_ID}_Sensor{TARGET_SENSOR_ID}.pdf')
plt.savefig(plot_pad, dpi=300, bbox_inches='tight')
plt.show()

### Cadansberekening (SPM) op Zuivere Lineaire Loopsegmenten

In [ ]:
def style_axes(ax):
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, direction='out', top=False, right=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#1a1a1a')
    ax.spines['bottom'].set_color('#1a1a1a')
    ax.grid(True, which='both', linestyle='-', color='#B0B0B0', linewidth=0.4, alpha=0.5)

FONT_SIZE_LABELS = 21
FONT_SIZE_TICKS = 19
FONT_SIZE_TITLE = 24

x_coords = np.array(runner_data[TARGET_AI_ID]['x_coords'])
y_coords = np.array(runner_data[TARGET_AI_ID]['y_coords'])
frames = np.array(runner_data[TARGET_AI_ID]['frames'])
time_stamps_ai = frames / FS_AI
rel_time_ai = time_stamps_ai - time_stamps_ai[0]

X_LEFT_LIMIT = 750
X_RIGHT_LIMIT = 2750

fig1, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot(rel_time_ai, x_coords, color='#2c3e50', linewidth=1.5, label='X-positie loper')
ax1.axhline(X_LEFT_LIMIT, color='#d62728', linestyle='--', linewidth=1.2, label=f'Linkergrens ({X_LEFT_LIMIT})')
ax1.axhline(X_RIGHT_LIMIT, color='#d62728', linestyle='--', linewidth=1.2, label=f'Rechtergrens ({X_RIGHT_LIMIT})')
ax1.set_title(f'Looprichting (X) over tijd voor Tracker #{TARGET_AI_ID}', fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=10)
ax1.set_xlabel('Tijd (s)', fontsize=FONT_SIZE_LABELS)
ax1.set_ylabel('X-positie', fontsize=FONT_SIZE_LABELS)
style_axes(ax1)
plt.tight_layout()
plt.show()

# Maskeren
valid_mask = (x_coords > X_LEFT_LIMIT) & (x_coords < X_RIGHT_LIMIT)
valid_indices = np.where(valid_mask)[0]

segments = []
for k, g in groupby(enumerate(valid_indices), lambda ix : ix[0] - ix[1]):
    segment = list(map(itemgetter(1), g))
    if len(segment) > (FS_AI * 2.5):
        segments.append(segment)

print(f"{len(segments)} bruikbare rechte loopsegmenten gevonden!")

segment_spms = []
fig2, ax2 = plt.subplots(figsize=(15, 7))

for i, seg in enumerate(segments):
    seg_t = rel_time_ai[seg]
    seg_y = y_coords[seg]
    seg_y_f = filtfilt(b_ai, a_ai, seg_y)
    peaks, _ = find_peaks(seg_y_f, distance=FS_AI*0.4)

    if len(peaks) > 1:
        time_diff = seg_t[peaks[-1]] - seg_t[peaks[0]]
        spm = ((len(peaks) - 1) * 2 / time_diff) * 60
        segment_spms.append(spm)
        p = ax2.plot(seg_t, seg_y_f, linewidth=1.5, label=f'Recht Segment {i+1}')
        ax2.scatter(seg_t[peaks], seg_y_f[peaks], color=p[0].get_color(), s=30, edgecolor='white', linewidth=0.8, zorder=5)

final_spm_title = np.mean(segment_spms) if segment_spms else 0.0

ax2.set_title(f"Head sway-analyse zonder bochten – loper #{TARGET_AI_ID}\nBerekende cadans: {final_spm_title:.1f}".replace('.', ',') + " SPM", fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=10)
ax2.set_xlabel('Tijd (s)', fontsize=FONT_SIZE_LABELS)
ax2.set_ylabel('Y-oscillatie (pixels)', fontsize=FONT_SIZE_LABELS)
ax2.set_ylim(-15, 15)
ax2.set_xlim(0, max(rel_time_ai) if len(rel_time_ai) > 0 else 100)
style_axes(ax2)
plt.tight_layout()
plt.show()

if segment_spms:
    final_spm = np.mean(segment_spms)
    print(f"\nGECORRIGEERDE AI SPM (Puur Lineair Loopgedrag): {final_spm:.1f} SPM")
else:
    print("Geen geldige segmenten gevonden.")